# Chapter 7 — Data Warehouse

**Source:** https://de101.startdataengineering.com/dw  
**Module:** Data Modeling

> **A data warehouse is a database that stores all your company's historical data.**

This chapter is conceptual — no code exercises. The goal is to understand *why* data warehouses exist, how they differ from transactional systems, and how column-oriented storage gives them their performance advantage.

---

## Why a warehouse?

As a business grows, analytical questions become harder to answer against the production database:

- Who are the top 10 sellers by items sold?
- What is the average order fulfillment time?
- Which customers buy similar items? (segmentation)
- Show a seller dashboard with top-performing items

These queries scan **millions of historical rows**. Running them on an OLTP database:
- Competes with live user traffic for I/O
- Loads irrelevant columns for every query
- Has no historical record (current state only)

A warehouse solves all three problems.

---

## 7.1 OLTP vs OLAP

| | OLTP | OLAP |
|---|---|---|
| **Stands for** | Online Transaction Processing | Online Analytical Processing |
| **Usage pattern** | Fast CRUD on a small number of rows | `SELECT ... GROUP BY` on millions of rows; bulk imports |
| **Storage type** | Row-oriented | Column-oriented |
| **Data modeling** | Normalization | Denormalization (dimensional modeling, data vaults) |
| **Data state** | Current state | Historical events |
| **Data size** | GB → TB | TB and above |
| **Examples** | MySQL, PostgreSQL | Clickhouse, Redshift, Snowflake, BigQuery |

### The core tension

OLTP is optimised for **writes** — fast, isolated row-level operations. A customer places an order → one row inserted, maybe a few rows updated. Normalization minimises redundancy so writes stay consistent.

OLAP is optimised for **reads at scale** — scanning millions of rows and aggregating. Denormalization pre-joins tables so the query engine doesn't have to do it at runtime. This makes writes slower but reads dramatically faster.

> **Rule:** never run heavy analytics on your production OLTP database. It degrades write performance for live users and still won't be fast enough for complex queries at scale.

**Note on Apache Spark:** Spark started as a pure processing engine (no storage). Over time it added table management (via Iceberg, Delta, Hudi) — effectively becoming an OLAP-adjacent system. This is why this course runs analytical queries on Spark.

---

## 7.2 Column-oriented storage — the mechanics

The single biggest performance difference between OLTP and OLAP comes from how data is physically laid out on disk.

**Key principle:** data is stored as continuous *pages* (groups of records) on disk. The engine reads full pages into memory — it cannot read half a page.

### Example table: `items`

| item_id | item_name | item_type | item_price | datetime_created | datetime_updated |
|---------|-----------|-----------|------------|-----------------|------------------|
| 1 | item_1 | gaming | 10 | 2021-10-02 00:00 | 2021-11-02 13:00 |
| 2 | item_2 | gaming | 20 | 2021-10-02 01:00 | 2021-11-02 14:00 |
| 3 | item_3 | biking | 30 | 2021-10-02 02:00 | 2021-11-02 15:00 |
| 4 | item_4 | surfing | 40 | 2021-10-02 03:00 | 2021-11-02 16:00 |
| 5 | item_5 | biking | 50 | 2021-10-02 04:00 | 2021-11-02 17:00 |

6 columns, 5 rows. Let's see how each storage type lays this out.

### Row-oriented layout (OLTP)

One row per page:

```
Page 1: [1, item_1, gaming,  10, '2021-10-02 00:00:00', '2021-11-02 13:00:00']
Page 2: [2, item_2, gaming,  20, '2021-10-02 01:00:00', '2021-11-02 14:00:00']
Page 3: [3, item_3, biking,  30, '2021-10-02 02:00:00', '2021-11-02 15:00:00']
Page 4: [4, item_4, surfing, 40, '2021-10-02 03:00:00', '2021-11-02 16:00:00']
Page 5: [5, item_5, biking,  50, '2021-10-02 04:00:00', '2021-11-02 17:00:00']
```

**Why this is great for OLTP:**  
Fetching a single customer record? Load 1 page → you have all columns for that row immediately. Fast for `SELECT * FROM orders WHERE order_id = 12345`.

**Why this is terrible for analytics:**  
To `SUM(item_price) GROUP BY item_type`, you still load all 5 pages — even though you only care about 2 of the 6 columns. The `item_name`, `datetime_created`, and `datetime_updated` columns are loaded and thrown away immediately.

On a real table with 10 million rows and 50 columns, you might only need 3 columns — but you load all 50.

### Column-oriented layout (OLAP)

One column per page:

```
Page 1 (item_id):          [1, 2, 3, 4, 5]
Page 2 (item_name):        [item_1, item_2, item_3, item_4, item_5]
Page 3 (item_type):        [gaming, gaming, biking, surfing, biking]
Page 4 (item_price):       [10, 20, 30, 40, 50]
Page 5 (datetime_created): ['2021-10-02 00:00:00', '2021-10-02 01:00:00', ...]
Page 6 (datetime_updated): ['2021-11-02 13:00:00', '2021-11-02 14:00:00', ...]
```

**The same query:**

```sql
SELECT item_type, SUM(item_price) AS total_price
FROM items
GROUP BY item_type;
```

| | Row-oriented | Column-oriented |
|---|---|---|
| Pages loaded | **5** (all rows) | **2** (page 3 + page 4 only) |
| Columns read into memory | 6 | 2 |
| I/O wasted | 4/6 = 67% irrelevant | 0% irrelevant |

This is **column pruning** — the engine skips pages for columns the query doesn't touch entirely.

At scale (200 columns, 3 needed for a query): column-oriented reads ~1.5% of what row-oriented reads.

### Two bonus advantages of column storage

**1. Better compression**

Column pages store the same data type, and often the same values, consecutively:

```
Page 3 (item_type): [gaming, gaming, biking, surfing, biking]
```

This compresses extremely well:
- **Run-length encoding:** `gaming ×2, biking ×1, surfing ×1, biking ×1` → 5 values → ~5 bytes instead of 40
- **Dictionary encoding:** replace strings with integer codes → `[0, 0, 1, 2, 1]` where `{0: gaming, 1: biking, 2: surfing}`

Row-oriented pages mix types (`int, string, string, float, datetime, datetime`) — much harder to compress.

**2. Vectorized processing**

Modern CPUs have SIMD (Single Instruction, Multiple Data) instructions that apply the same operation to a batch of values in a single clock cycle. Column pages are a perfect fit — a page of integers can be summed with SIMD, operating on 8–16 values per CPU instruction instead of 1.

Row pages cannot use SIMD — each position has a different type.

---

**Summary: why OLAP is dramatically faster for analytics**

| Advantage | Mechanism | Benefit |
|---|---|---|
| Column pruning | Skip irrelevant column pages entirely | Less I/O |
| Better compression | Same-type, similar-value sequences compress well | Less storage + less I/O |
| Vectorized processing | SIMD on uniform-type column batches | More compute per clock |
| Bulk insert | Append-only writes, no random I/O | Fast ingest |

All four advantages compound together — this is why Snowflake/BigQuery/Redshift can scan billions of rows in seconds.

---

## 7.3 Compression algorithms — the mechanics

Columnar storage enables aggressive compression because each page holds a single data type with often-repetitive values. Engines like Parquet, ORC, Snowflake, and BigQuery apply several algorithms, sometimes stacked.

---

### Algorithm 1 — Run-Length Encoding (RLE)

**Best for:** low-cardinality columns with long consecutive runs (e.g. `item_type`, `country`, `status`).

Instead of storing every value, store *(value, repeat_count)* pairs.

```
Raw:         [gaming, gaming, gaming, biking, biking, surfing, biking, biking, biking, biking]
RLE encoded: [(gaming, 3), (biking, 2), (surfing, 1), (biking, 4)]

Raw bytes:   10 × 6 bytes avg = 60 bytes
RLE bytes:   4 × (6 + 1) bytes = 28 bytes   →  53% smaller
```

RLE is most powerful when data is **sorted** — values cluster, making runs longer. This is why warehouse tables are often sorted on low-cardinality columns (e.g., `ORDER BY country, status`).

---

### Algorithm 2 — Dictionary Encoding

**Best for:** string columns with a bounded set of distinct values (enums, categories, IDs).

Build a lookup table (the "dictionary"), then store integer codes instead of strings.

```
Dictionary:  {0: "gaming", 1: "biking", 2: "surfing"}
Raw column:  ["gaming", "gaming", "biking", "surfing", "biking"]   → 5 × ~6 bytes = 30 bytes
Encoded:     [0, 0, 1, 2, 1]                                       → 5 × 1 byte  =  5 bytes + 3-entry dict

Net saving: ~80% on the column values
```

Dictionary encoding is applied first. After encoding, RLE is often applied on top of the integer codes — the two stack very effectively:

```
Dictionary codes:  [0, 0, 0, 1, 1, 2, 1, 1, 1, 1]
After RLE:         [(0,3), (1,2), (2,1), (1,4)]
```

**Note:** Snowflake and BigQuery use dictionary + RLE as their default for string and low-cardinality columns. You don't configure this — it's automatic.

---

### Algorithm 3 — Delta Encoding

**Best for:** monotonically increasing numeric sequences (timestamps, IDs, dates).

Store only the *difference* between consecutive values, not the values themselves.

```
Raw timestamps:   [1000000, 1000060, 1000120, 1000180, 1000240]
Deltas:           [1000000,      60,      60,      60,      60]
                   (base)    (+60s)   (+60s)   (+60s)   (+60s)

Raw:    5 × 8 bytes = 40 bytes
Delta:  1 × 8 bytes (base) + 4 × 1 byte (small deltas) = 12 bytes   →  70% smaller
```

After delta encoding, RLE often applies again (repeated delta of 60 becomes one entry). Parquet uses this automatically for `TIMESTAMP` and `INT64` columns.

---

### Algorithm 4 — Bit-Packing

**Best for:** integer columns where values fit in fewer bits than the declared type.

If a column is declared `INT64` (8 bytes) but values only range 0–15 (4 bits), you can pack 16 values into a single 8-byte word.

```
INT64 column [0, 3, 7, 15, 2, ...]:
Normal:   each value = 64 bits
Packed:   each value = 4 bits → 16× compression on the column
```

Bit-packing is often combined with delta encoding: delta first (to shrink range), then pack the small integers tightly.

---

### How engines choose algorithms automatically

| Column type | Likely algorithm(s) applied |
|---|---|
| Low-cardinality string (`status`, `country`) | Dictionary → RLE |
| High-cardinality string (`name`, `email`) | Dictionary (no RLE benefit) |
| Monotonic timestamps/IDs | Delta → Bit-packing → RLE |
| Sparse integers (large range) | Bit-packing |
| Boolean / NULL flags | Bit-packing (1 bit per value) |
| Floats (prices, metrics) | Usually just dictionary or no compression |

The engine profiles each column and chooses the best algorithm per column-chunk — you never configure this manually in Snowflake, BigQuery, or Parquet with default settings.

---

## 7.4 Storage file formats — Parquet, ORC, Avro

The physical file format is where columnar storage actually lives on disk. Understanding formats is essential — they determine what optimisations are possible and which tools can read your data.

---

### Apache Parquet

The dominant format for analytical workloads. Used by Spark, Athena, BigQuery (external tables), Snowflake external stages, dbt, and almost every modern data tool.

**Physical structure:**

```
Parquet File
├── Row Group 1  (e.g. 128MB chunk of rows)
│   ├── Column Chunk: item_id       [page | page | page ...]
│   ├── Column Chunk: item_type     [page | page | page ...]  ← dictionary + RLE encoded
│   ├── Column Chunk: item_price    [page | page | page ...]
│   └── Column statistics: min/max/null_count per column chunk
├── Row Group 2
│   └── ...
└── Footer (file-level metadata: schema, row group offsets, statistics)
```

Key design decisions:
- **Row groups** allow partial file reads — to answer a query, the engine reads only the row groups that might contain matching data (using min/max stats)
- **Column chunks** within a row group are contiguous — column pruning skips entire chunks
- **Page-level encoding** — each page (~1MB) is independently compressed and encoded
- **Footer metadata** — schema + statistics are stored at the end; the engine reads the footer first, then decides which row groups and columns to actually fetch

**The statistics trick (predicate pushdown):**

```sql
SELECT * FROM orders WHERE order_date >= '2024-01-01'
```

Parquet stores `min(order_date)` and `max(order_date)` per row group. If a row group has `max = '2023-12-31'`, the engine **skips it entirely without reading any data**. This is called *predicate pushdown* or *filter pushdown*.

---

### Apache ORC (Optimized Row Columnar)

ORC is Parquet's main competitor, historically preferred in the Hive ecosystem. Both are columnar, but differ in detail:

| | Parquet | ORC |
|---|---|---|
| Ecosystem | Spark, Athena, dbt, Iceberg | Hive, older Spark, Presto |
| Default compression | Snappy / Zstandard | Zlib / Snappy |
| Stats granularity | Column chunk (row group level) | Stripe + index (finer-grained) |
| Nested types | Excellent (Dremel encoding) | Good |
| Default in | Iceberg, Delta Lake | Hive tables |

**Rule of thumb:** use Parquet unless you're working in a Hive-native environment. Parquet has won the ecosystem battle.

---

### Apache Avro

Avro is **row-oriented** — it's the odd one out in this list. It stores rows, not columns.

**Why does it exist?**

| Use case | Best format |
|---|---|
| Analytical queries (aggregations, GROUP BY) | Parquet or ORC |
| Streaming / event logs (Kafka messages) | Avro |
| Schema evolution (fields added/removed over time) | Avro |
| Write-heavy, read-once (landing zone) | Avro |
| Full row retrieval (export, replication) | Avro |

Avro is excellent for **schema registry** use with Kafka — every message carries a schema reference, enabling robust schema evolution (add nullable fields without breaking consumers). In data pipelines:

```
Kafka (Avro) → Landing Zone → Transform → Parquet (warehouse)
```

Avro is the transport format; Parquet is the storage format.

---

### Format selection cheat sheet

```
Is the data for streaming / event transport?       → Avro
Is it for analytics / GROUP BY queries?            → Parquet
Is it in a Hive-native stack?                      → ORC
Is it going into Iceberg or Delta Lake?            → Parquet (default)
Is it a landing/staging zone before transforms?    → Avro or Parquet (both fine)
```

---

### How Parquet fits with Iceberg (what you're using in this course)

This course runs on Spark + Iceberg. Iceberg stores its data files as **Parquet** by default. Iceberg adds:
- A **metadata layer** (JSON manifest files) on top of Parquet files
- ACID transactions (atomic commits, snapshot isolation)
- Schema evolution and partition evolution
- Time travel (query a table as it was at a past snapshot)

```
Iceberg table
├── metadata/
│   ├── v1.metadata.json    ← table schema, partition spec, snapshot history
│   ├── snap-001.avro       ← manifest list (which data files belong to this snapshot)
│   └── manifest-001.avro  ← manifest (stats + paths for each Parquet file)
└── data/
    ├── part-00001.parquet
    ├── part-00002.parquet
    └── ...
```

The metadata itself (manifest files) is stored in Avro — a good example of Avro in the write-heavy landing role.

---

## 7.5 How a query actually executes in a columnar system

When you write `SELECT item_type, SUM(item_price) FROM items GROUP BY item_type`, what actually happens between submitting the query and getting results? Understanding this pipeline explains *why* certain queries are fast or slow — and how to make them faster.

---

### The query execution pipeline (end to end)

```
SQL query submitted
        ↓
1. PARSE         — tokenise and validate SQL syntax
        ↓
2. PLAN          — build a logical query plan (relational algebra tree)
        ↓
3. OPTIMISE      — rewrite the plan for efficiency (pushdowns, join reordering)
        ↓
4. SCHEDULE      — split into physical tasks, assign to workers/executors
        ↓
5. SCAN          — read only needed files, row groups, and columns from disk
        ↓
6. FILTER        — apply WHERE conditions as early as possible
        ↓
7. PROJECT       — select only the needed columns, drop the rest
        ↓
8. AGGREGATE     — GROUP BY, SUM, COUNT, etc.
        ↓
9. SHUFFLE       — redistribute data across workers if needed (expensive)
        ↓
10. RETURN       — send results back to the client
```

The optimiser (step 3) is where most of the magic happens. Three key optimisations:

---

### Optimisation 1 — Predicate Pushdown

Move filter conditions as close to the data source as possible — ideally into the file format itself.

```sql
SELECT item_type, SUM(item_price)
FROM items
WHERE order_date >= '2024-01-01'   ← this filter
GROUP BY item_type
```

**Without pushdown:** read all files → filter in memory → aggregate  
**With pushdown:** use Parquet row group statistics (`min/max order_date`) → skip entire row groups where `max(order_date) < '2024-01-01'` → filter the rest → aggregate

```
File A: row group min=2023-01-01, max=2023-06-30  → SKIPPED (entirely, no bytes read)
File B: row group min=2023-07-01, max=2024-03-31  → read and filter
File C: row group min=2024-01-01, max=2024-12-31  → read (all rows pass)
```

Predicate pushdown works at three levels:
1. **File level** — skip entire Parquet/ORC files using partition metadata
2. **Row group level** — skip row groups using min/max column statistics in the footer
3. **Page level** — skip pages using page-level statistics (Parquet v2)

This is the single most impactful query optimisation in a columnar warehouse. Well-designed tables can achieve 99%+ file skip rates for time-range queries.

---

### Optimisation 2 — Projection Pushdown (Column Pruning)

Only read the column chunks that the query touches — skip the rest entirely.

```sql
SELECT item_type, SUM(item_price)   ← only 2 columns needed
FROM items                          ← table has 6 columns
GROUP BY item_type
```

The engine reads only column chunks `item_type` and `item_price`. The `item_id`, `item_name`, `datetime_created`, `datetime_updated` column chunks are never fetched from disk.

On a 100-column fact table where a query needs 5 columns: **95% of disk I/O is eliminated before the query even starts.**

---

### Optimisation 3 — Late Materialisation

In columnar execution, values from different columns are only combined into rows at the very last moment — right before returning results.

```
Naive approach:
  Read item_type column → read item_price column → zip together into rows → filter → aggregate

Late materialisation:
  Read item_type column → evaluate filter → get list of matching row positions
  Read item_price column → fetch ONLY the positions that passed the filter
  Aggregate → return
```

If 90% of rows are filtered out by `WHERE item_type = 'gaming'`, late materialisation means only 10% of `item_price` values are ever read from disk. The filter on `item_type` acts as a position mask applied before other columns are fetched.

---

### What this means for query design

| Design decision | Why it matters |
|---|---|
| Put filter columns first in sort order | Enables row group skipping (predicate pushdown) |
| Partition on the most common filter column | Enables file-level skipping |
| Avoid `SELECT *` in production queries | Defeats projection pushdown — reads all columns |
| Filter early, aggregate late | The engine can push filters down; aggregations must happen after reads |
| Keep row groups clustered by date | Time-range queries skip entire row groups |

The query optimiser handles most of this automatically — but your table *design* (partitioning, sort order) determines the ceiling of what the optimiser can achieve.

---

## 7.6 Partitioning and clustering

Partitioning and clustering are the two main table design levers that let the query engine skip data before reading a single byte. They work at different granularities.

---

### Partitioning — file-level skipping

Partitioning splits a table into **separate directories/files** based on column values. The engine uses partition metadata to skip entire files before reading anything.

```
orders/
├── order_date=2024-01/
│   ├── part-001.parquet   (all January 2024 orders)
│   └── part-002.parquet
├── order_date=2024-02/
│   └── part-001.parquet   (all February 2024 orders)
└── order_date=2024-03/
    └── part-001.parquet
```

```sql
-- This query touches ONLY the 2024-02 partition directory
SELECT COUNT(*) FROM orders WHERE order_date >= '2024-02-01' AND order_date < '2024-03-01'
-- Files in 2024-01/ and 2024-03/ are never opened
```

**Partition column selection rules:**
- Choose columns that appear frequently in `WHERE` clauses
- Low-to-medium cardinality (date, region, status) — not high cardinality (user_id, order_id)
- Avoid over-partitioning: thousands of tiny partitions create metadata overhead ("small files problem")
- `order_date` (monthly or daily) is the most common partition key in analytics

**Over-partitioning anti-pattern:**

```
-- BAD: 10M distinct customer_ids → 10M partition directories → metadata explosion
PARTITIONED BY (customer_id)

-- GOOD: ~365 or ~120 partitions per year
PARTITIONED BY (DATE_TRUNC('month', order_date))
```

---

### Clustering / Sorting — row-group-level skipping

Within a partition (or a non-partitioned table), clustering defines the **sort order** of rows inside files. This enables predicate pushdown at the row group level using min/max statistics.

```sql
-- Table clustered by (region, order_date)
-- Row group 1: region IN ('EU'), order_date IN ['2024-01-01', '2024-01-31']
-- Row group 2: region IN ('EU'), order_date IN ['2024-02-01', '2024-02-28']
-- Row group 3: region IN ('US'), order_date IN ['2024-01-01', '2024-01-31']

-- Query: WHERE region = 'US' AND order_date = '2024-01-15'
-- Row groups 1 and 2 skipped entirely using min/max stats
-- Only row group 3 is read
```

Clustering also improves compression — sorted data creates longer runs for RLE and more compact dictionary codes.

**Partitioning vs Clustering at a glance:**

| | Partitioning | Clustering / Sorting |
|---|---|---|
| Granularity | File / directory level | Row group level (within a file) |
| Skipping mechanism | Partition catalog metadata | Min/max column statistics in Parquet footer |
| Best for | High-selectivity equality filters | Range filters + compression |
| Cost | File count overhead | Re-sort on write (one-time cost) |
| Maintenance | Partition pruning is free at query time | Z-ordering or OPTIMIZE commands (Delta/Iceberg) |

**Iceberg Z-ordering (multi-column clustering):**

Iceberg supports `OPTIMIZE ... ZORDER BY (col1, col2)` — a space-filling curve that co-locates rows with similar values across *multiple* columns into the same row groups, enabling multi-dimensional predicate pushdown. This is the production solution when you need to filter on multiple columns that don't have a natural total order.

---

### Practical rules for partitioning/clustering

```
1. Partition on the column that appears in WHERE most often → date is almost always first choice
2. Cluster on the next most common filter columns
3. Target row group size: 128MB–1GB (Parquet default is 128MB)
4. Avoid partitions smaller than 1GB total — too many small files
5. For time-series data: PARTITION BY month, CLUSTER BY (region, item_type)
6. After large writes: run OPTIMIZE / VACUUM to compact and sort files
```

---

## 7.7 Data Warehouse vs Data Lake vs Lakehouse

These three terms are often used loosely. As a data engineer you'll hear all three constantly — here's the precise distinction and when each makes sense.

---

### Data Warehouse

A **managed, schema-enforced, query-optimised** analytical database.

```
Characteristics:
✓ Schema-on-write — data validated and typed before storage
✓ SQL interface, ACID transactions
✓ Highly optimised query engine (columnar, vectorised)
✓ Built-in governance (access control, audit logs)
✗ Expensive at scale (proprietary compute + storage bundled)
✗ Closed format — data locked in vendor format
✗ Poor support for unstructured/semi-structured data (JSON blobs, images)

Examples: Snowflake, BigQuery, AWS Redshift, Azure Synapse
```

**When to use:** production BI/reporting, well-understood structured data, teams that need SQL without infrastructure management.

---

### Data Lake

Raw storage for **any format, any schema**, usually on cloud object storage (S3, GCS, ADLS).

```
Characteristics:
✓ Schema-on-read — schema inferred at query time
✓ Stores anything: Parquet, Avro, JSON, CSV, images, videos
✓ Extremely cheap storage (S3 ~$23/TB/month)
✓ Open format — any tool can read
✗ No ACID transactions (without a table format layer)
✗ No enforced schema — easy to write garbage, hard to read later
✗ "Data swamp" risk — data lands but is never organised or trusted
✗ Query performance varies widely depending on tooling

Examples: S3 + Athena, GCS + BigQuery External Tables, ADLS + Synapse
```

**When to use:** raw data landing zone, ML feature stores, archiving, event logs, data you don't yet know how to use.

---

### Lakehouse

A **table format layer** on top of a data lake that adds warehouse-like guarantees (ACID, schema, versioning) while keeping open formats.

```
Characteristics:
✓ ACID transactions on top of Parquet files (atomic commits)
✓ Schema enforcement + evolution
✓ Time travel (query data as it was at a previous snapshot)
✓ Open format (Parquet) — any engine can read
✓ Cheap storage (object store)
✓ Unified: same table serves both BI SQL and ML/Spark workloads
✗ More operational complexity than a managed warehouse
✗ Query performance depends on table maintenance (compaction, Z-ordering)

Examples: Apache Iceberg, Delta Lake, Apache Hudi
```

**This is what you're using in this course** — Iceberg on MinIO (S3-compatible) with Spark.

---

### The architecture spectrum

```
Raw Storage ←————————————————————————————→ Managed Warehouse

  S3/GCS         Data Lake      Lakehouse        Snowflake
  (files)    (Parquet + tools)  (Iceberg/Delta)  (full service)

  Cheapest                                        Most managed
  Most flexible                                   Most optimised
  No governance                                   Built-in governance
```

---

### How they fit together in a real pipeline

Most mature data platforms use all three:

```
Source systems (OLTP)
        ↓
  Landing Zone (Data Lake — raw Avro/JSON in S3)
        ↓
  Cleansed Layer (Lakehouse — Iceberg/Delta Parquet)
        ↓
  Warehouse Layer (Snowflake/BigQuery — curated, governed, fast)
        ↓
  BI Tools (Tableau, Looker, Metabase)
```

The Lakehouse layer is often called the "Silver" or "Gold" layer — more on this in Chapter 9 (multi-hop architecture).

---

### ACID on top of object storage — how Iceberg achieves it

Object storage (S3) has no native ACID support — you can't atomically update a file. Iceberg solves this with **optimistic concurrency control** via its metadata layer:

```
Write transaction:
1. Writer computes new Parquet data files and writes them to S3 (no commit yet)
2. Writer creates a new manifest file listing the new data files
3. Writer atomically swaps the metadata pointer to the new snapshot
   (using S3 conditional put / atomic rename where available)
4. If the swap fails (another writer committed first) → retry or abort

Read at any time:
- Reader follows the current metadata pointer → gets a consistent snapshot
- Old snapshots remain available → time travel
```

This is why Iceberg tables on S3 can have ACID semantics: the atomicity comes from the metadata swap, not from locking individual data files.

---

## 7.8 OLTP vs OLAP schema design — concrete SQL contrast

The same business data looks radically different depending on whether it's in an OLTP or OLAP system. This section shows both designs for the same domain (an e-commerce bike parts store) so you can feel the difference concretely.

---

### OLTP schema (normalised, 3NF)

Designed for write correctness. Every fact lives in one place — updates are fast and consistent.

```sql
-- Customers table — source of truth for customer info
CREATE TABLE customers (
    customer_id   INT          PRIMARY KEY,
    name          VARCHAR(100) NOT NULL,
    email         VARCHAR(150) UNIQUE NOT NULL,
    country       VARCHAR(50),
    created_at    TIMESTAMP    DEFAULT NOW()
);

-- Items table — source of truth for item info
CREATE TABLE items (
    item_id       INT          PRIMARY KEY,
    name          VARCHAR(100) NOT NULL,
    item_type     VARCHAR(50),
    price         DECIMAL(10,2),
    updated_at    TIMESTAMP
);

-- Orders table — source of truth for order header
CREATE TABLE orders (
    order_id      INT          PRIMARY KEY,
    customer_id   INT          REFERENCES customers(customer_id),
    order_date    DATE         NOT NULL,
    status        VARCHAR(20)  DEFAULT 'pending'
);

-- Order lines — each item in an order
CREATE TABLE order_lines (
    line_id       INT          PRIMARY KEY,
    order_id      INT          REFERENCES orders(order_id),
    item_id       INT          REFERENCES items(item_id),
    quantity      INT,
    unit_price    DECIMAL(10,2)  -- snapshot of price at time of order
);
```

**To answer "what did customer 42 buy last month?":**

```sql
SELECT i.name, ol.quantity, ol.unit_price
FROM orders o
JOIN order_lines ol ON o.order_id = ol.order_id
JOIN items i        ON ol.item_id = i.item_id
WHERE o.customer_id = 42
  AND o.order_date >= CURRENT_DATE - INTERVAL '30 days';
-- 3 joins required for a simple lookup
```

**Pros:** updates to `customer.email` touch 1 row, not millions of order records. No redundancy. Fast writes.  
**Cons:** every analytical query requires multiple joins across normalised tables.

---

### OLAP schema (denormalised, star schema)

Designed for read performance. Data is pre-joined and flattened so queries touch fewer tables.

```sql
-- Fact table — one row per order line, all keys and measures here
CREATE TABLE fact_order_lines (
    order_line_id   BIGINT,
    order_id        BIGINT,
    order_date      DATE,         -- denormalised from orders
    customer_id     BIGINT,
    item_id         BIGINT,
    quantity        INT,
    unit_price      DECIMAL(10,2),
    total_price     DECIMAL(10,2) -- pre-computed: quantity × unit_price
)
PARTITIONED BY (order_date);

-- Dimension table — customer attributes at time of analysis
CREATE TABLE dim_customer (
    customer_id   BIGINT,
    name          VARCHAR(100),
    email         VARCHAR(150),
    country       VARCHAR(50),
    customer_segment VARCHAR(20)  -- derived/enriched field
);

-- Dimension table — item attributes
CREATE TABLE dim_item (
    item_id       BIGINT,
    name          VARCHAR(100),
    item_type     VARCHAR(50),
    price_tier    VARCHAR(20)     -- derived: 'budget', 'mid', 'premium'
);
```

**The same analytical question — now simpler:**

```sql
-- "Total revenue by item type for EU customers in Q1 2024"
SELECT
    di.item_type,
    SUM(fol.total_price) AS revenue
FROM fact_order_lines fol
JOIN dim_item     di ON fol.item_id     = di.item_id
JOIN dim_customer dc ON fol.customer_id = dc.customer_id
WHERE fol.order_date BETWEEN '2024-01-01' AND '2024-03-31'
  AND dc.country IN ('DE', 'FR', 'NL', 'ES', 'IT')
GROUP BY di.item_type
ORDER BY revenue DESC;

-- 2 joins (dimension lookups), partition pruning on order_date
-- vs 3+ joins on the normalised OLTP schema
```

---

### Side-by-side design comparison

| Concern | OLTP (normalised) | OLAP (denormalised) |
|---|---|---|
| Update `customer.email` | 1 row changed | Need to re-run ETL to propagate |
| Count orders per country | 3 JOINs | 1 JOIN (country in fact or dim) |
| Add new derived column | Schema migration | Add computed column in dbt model |
| Historical price at order time | Stored in `order_lines.unit_price` | Stored in `fact_order_lines.unit_price` |
| Data redundancy | Minimal | High (country repeated on every row) |
| Query speed at 1B rows | Slow (many joins) | Fast (fewer joins, partition pruning) |

---

### The golden rule

> **Normalise for writes. Denormalise for reads.**

OLTP and OLAP are not competing approaches — they're complementary layers. The OLTP system guarantees data integrity; the ETL pipeline moves data into the warehouse where it's denormalised for analytical speed. This is the fundamental pattern of every modern data platform.

---

## 7.9 — The Metadata Layer: Catalog and Iceberg system tables

Every warehouse needs something to track: where are the files? What schema do the columns have? Which partitions exist? Which files belong to which snapshot? This is the catalog's job. Without it, every query would need to list all files and read all footers cold.

In this course, Spark uses an **Iceberg REST catalog** backed by MinIO. The catalog exposes table metadata as queryable system tables you can inspect directly.

In [ ]:
# --- Inspect the catalog configuration Spark is using ---
print("Default catalog:", spark.conf.get("spark.sql.defaultCatalog"))
print("Warehouse location:", spark.conf.get("spark.sql.catalog.local.warehouse"))
print("Catalog type:", spark.conf.get("spark.sql.catalog.local.type", "not set"))

In [ ]:
# --- Table properties: see Iceberg format version and other metadata stored in the catalog ---
spark.sql("SHOW TBLPROPERTIES prod_db.orders").show(truncate=False)

**What `SHOW TBLPROPERTIES` reveals:**

- `format-version` = `2` — Iceberg v2, which supports row-level delete files (the Merge-on-Read pattern for updates/deletes)
- `write.format.default` = `parquet` — data files are Parquet
- `write.parquet.compression-codec` — the codec used (Snappy by default)

These properties live in the catalog metadata layer, not in the Parquet files themselves.

In [ ]:
# --- Iceberg snapshot history — every write creates an immutable snapshot ---
# This is the mechanism behind time travel
spark.sql("SELECT * FROM prod_db.orders.history").show(truncate=False)

In [ ]:
# --- Iceberg files metadata — the physical Parquet files the catalog tracks ---
# Each row = one Parquet file. Note: record_count, file_size_in_bytes,
# and column-level min/max statistics are stored HERE (in the manifest), not just in the Parquet footer.
# This is what enables the catalog to skip files without opening them.
spark.sql("""
    SELECT 
        file_path,
        record_count,
        file_size_in_bytes,
        column_sizes
    FROM prod_db.orders.files
    LIMIT 5
""").show(truncate=False)

**The three-layer Iceberg metadata chain:**

```
catalog pointer
      ↓
snapshot (snapshot_id, timestamp, operation)
      ↓
manifest list → manifest files (list of data files + their column stats)
      ↓
data files (Parquet)
```

Every query starts at the catalog pointer and walks down this chain to find the right Parquet files. The column statistics in the manifest files are what enable file-level predicate pushdown — the engine reads the manifest, checks stats, and may skip the Parquet file entirely without opening it.

The `column_sizes` map in `prod_db.orders.files` shows bytes per column — this is a live example of the column-oriented storage you just learned about. Each column in each Parquet file has independently tracked metadata.

---

## 7.10 — Predicate pushdown: observing it in action

The concepts of predicate pushdown and projection pushdown are visible in Spark's `EXPLAIN` output. Let's run the query from section 7.5 against the real TPC-H data and read the physical plan.

In [ ]:
## Run the analytical query first — see actual results
# This is the "average fulfillment time by shipping mode" from the business questions in section 7.1

query = """
    SELECT 
        l_shipmode,
        ROUND(AVG(DATEDIFF(l_receiptdate, l_shipdate)), 2) AS avg_days_to_receipt,
        MIN(DATEDIFF(l_receiptdate, l_shipdate))           AS min_days,
        MAX(DATEDIFF(l_receiptdate, l_shipdate))           AS max_days,
        COUNT(*)                                           AS total_shipments
    FROM prod_db.lineitem
    WHERE l_shipdate BETWEEN '1993-01-01' AND '1997-12-31'
    GROUP BY l_shipmode
    ORDER BY avg_days_to_receipt
"""

spark.sql(query).show()

In [ ]:
# --- Now read the physical execution plan ---
# Look for three key fields that prove pushdown is happening:
#
#   PushedFilters:   filters the Parquet reader applies BEFORE rows enter Spark memory
#   ReadSchema:      only the columns listed here are read from disk (projection pushdown)
#   PartitionFilters: file-level skipping (if table is partitioned by shipdate)
#
# If PushedFilters is non-empty → predicate pushdown is active
# If ReadSchema is a subset of all columns → projection pushdown is active

spark.sql(query).explain("formatted")

**How to read the EXPLAIN output:**

The bottom of the plan contains the `BatchScan` or `IcebergScan` node — this is where data enters the pipeline. Look for:

```
(N) IcebergScan [...]
    PushedFilters: [IsNotNull(l_shipdate), GreaterThanOrEqual(l_shipdate, ...), LessThanOrEqual(l_shipdate, ...)]
    ReadSchema: struct<l_shipmode:string, l_shipdate:date, l_receiptdate:date>
```

- **`PushedFilters`** — these filters run inside the Parquet reader, before any data reaches Spark executors. Rows that don't pass are never deserialised. This is predicate pushdown.
- **`ReadSchema`** — only 3 of the 16 `lineitem` columns are being read from disk. The other 13 column chunks in every Parquet file are skipped entirely. This is projection pushdown.

The plan reads bottom-up. Data flows: `IcebergScan → Filter → Project → HashAggregate → Sort → return`

At the scan level, both optimisations are already applied — the data arriving at the `Filter` node is already pre-filtered and column-pruned by the Parquet reader.

---

## 7.11 — Real-world sizing: when does each tool make sense?

A common mistake is reaching for Spark or Snowflake when the data actually fits in a laptop. Here is a practical decision framework based on data volume and team needs.

| Data size | Row count (est.) | Recommended tool | Why |
|---|---|---|---|
| < 1 GB | < 10M rows | **DuckDB** or pandas | Runs on any machine in seconds. Free. No infra. |
| 1 GB – 100 GB | 10M – 500M rows | **DuckDB on Parquet** | Single machine columnar SQL. Still free. Most companies are here. |
| 100 GB – 1 TB | 500M – 5B rows | **Managed warehouse** (Snowflake/BigQuery) or Spark on small cluster | Engineering cost of operating Spark cluster starts to exceed warehouse subscription cost |
| 1 TB – 100 TB | 5B – 500B rows | **Spark + Iceberg on S3** | Storage cost difference vs managed warehouse justifies cluster ops |
| 100 TB+ | 500B+ rows | Spark + Iceberg, tiered storage, aggressive pruning | Full-time engineering concern — partitioning and compaction become critical |

**The most underused tool in data engineering: DuckDB**

DuckDB is an embedded columnar SQL engine that reads Parquet files directly. It has no server, no setup, no cost. It is faster than Spark for data that fits on one machine — because Spark's distributed overhead (scheduling, serialization, shuffle) dominates at small scale.

Run the cell below to prove it: DuckDB queries the same TPC-H data that the entire course uses Spark for — in milliseconds, from a Python import.

In [ ]:
import duckdb
import time

conn = duckdb.connect(":memory:")

# Query the TPC-H CSV data directly — no loading, no setup
# DuckDB scans CSVs with its own vectorised columnar engine
start = time.time()

result = conn.execute("""
    SELECT 
        o_orderstatus,
        COUNT(*)            AS order_count,
        ROUND(SUM(o_totalprice), 2) AS total_revenue
    FROM '/home/airflow/notebooks/sample_data/orders.csv'
    GROUP BY o_orderstatus
    ORDER BY total_revenue DESC
""").df()

elapsed = time.time() - start

print(result.to_string(index=False))
print(f"\nQuery completed in {elapsed:.3f}s using DuckDB (no Spark, no cluster, no infra)")
print(f"This data would be in Snowflake / Spark only if it exceeded ~100GB")

**Why we use Spark in this course despite the data being small:**

Pedagogical choice — the tools and patterns you're learning (Spark, Iceberg, dbt, Airflow) are what you'll use in production on large datasets. The TPC-H data at scale factor 0.1 is small enough to run on a single container, but every query, every `EXPLAIN`, every partition and Iceberg concept transfers directly to TB-scale workloads.

**Rule for your own work:** start with DuckDB. Graduate to Spark + Iceberg when either (a) data genuinely can't fit on one machine or (b) you need ACID multi-writer semantics in a team setting.

---

## Key Takeaways

**Fundamentals**
- **OLTP ≠ OLAP** — same SQL syntax, completely different physical design. Pick the right system for the workload.
- **Row vs column storage is the fundamental split** — row is fast for writes and point lookups; column is fast for aggregating across millions of rows.
- **Warehouses preserve history** — OLTP shows current state; warehouses accumulate every historical event.
- **Normalise for writes. Denormalise for reads.** OLTP and OLAP are complementary, not competing.

**Compression**
- **Algorithms stack** — Dictionary → RLE → Bit-packing are applied per column chunk automatically. You don't configure this.
- **Sort order unlocks RLE compression** — sorted data creates longer runs; clustering and partitioning both drive this.
- **Delta encoding for timestamps** — stores differences, not absolute values; combines with bit-packing for tiny deltas.

**File formats**
- **Parquet is the default** — use it for analytics. ORC for Hive-native stacks. Avro for streaming/transport/Kafka.
- **Parquet row groups = unit of I/O skipping** — column pruning skips columns; row group stats skip row ranges; partitions skip files. Three independent layers.
- **Parquet footer = metadata** — schema, row group offsets, min/max stats. The engine reads this first before any data.
- **Iceberg = Parquet + metadata layer** — adds ACID, time travel, and schema evolution. The data is still Parquet.

**Query execution**
- **Predicate pushdown is the biggest win** — `PushedFilters` in `EXPLAIN` output proves it. Filters run inside the Parquet reader before rows enter Spark.
- **Projection pushdown (column pruning)** — `ReadSchema` in `EXPLAIN` output shows only the needed columns are read. On a 16-column table, a 3-column query reads 3/16 of the disk I/O.
- **Late materialisation** — column values combined into rows only after filters applied. A 90% filter means only 10% of other columns are fetched.
- **`SELECT *` defeats column pruning** — never use it in production analytical queries.

**Partitioning and clustering**
- **Partition on the most common filter column** — almost always a date. File-level skipping, no I/O cost at query time.
- **Cluster/sort within partitions** — enables row group skipping via min/max stats. Unsorted = stats span full range = no skipping.
- **Small files problem** — over-partitioning (e.g. by user_id) creates millions of tiny files. Use OPTIMIZE/compaction.

**Catalog and metadata**
- **The catalog is as important as the data** — it stores schema, partition info, file paths, and statistics. Without it, planning is blind.
- **Iceberg exposes metadata as queryable tables** — `.history`, `.files`, `.snapshots` are real SQL queries you can run.
- **Manifest stats enable file-level skipping** — Iceberg stores column min/max in manifests, so the engine may skip a Parquet file without opening it.

**Warehouse vs Lake vs Lakehouse**
- **Lake** = cheap, open, ungoverned. **Warehouse** = fast, managed, expensive. **Lakehouse** = open format + ACID + cheap storage.
- **ACID on object storage** = optimistic concurrency via atomic metadata pointer swap (Iceberg), not file locking.
- **Iceberg is a transaction layer, not a storage format** — data is still Parquet; Iceberg adds the commit protocol on top.

**Sizing**
- **Use DuckDB before you reach for Spark** — under 100GB, DuckDB is faster, simpler, and free. Spark's distributed overhead dominates at small scale.
- **Spark is the right tool when data can't fit on one machine** — or when ACID multi-writer semantics in a team setting are needed.

---

## Next: Chapter 8

Chapter 8 covers **Kimball dimensional modeling** — the dominant design pattern for warehouse tables.

Two table types:
- **Fact tables** — measurements (orders, events, transactions) — wide, many rows
- **Dimension tables** — context (customers, products, sellers) — narrower, fewer rows

Source: https://de101.startdataengineering.com/dw_table_types